In [1]:
import matplotlib.pyplot as plt
import os
import glob
import pandas_datareader.data as web
import datetime
import pandas as pd
from dbnomics import fetch_series
from utils import fetch_imf_data, fetch_imf_bulk
from config import country_dict, us_economy_metrics, country_dict_3_char


In [2]:
# Paths
imf_cpi_path = "data/imf_cpi.csv"
imf_gdp_path = "data/imf_gdp.csv"
imf_unemployment_path = "data/imf_unemployment_rate.csv"
imf_industrial_production_index_path = "data/imf_industrial_production_index.csv"
imf_exports_path = "data/imf_exports.csv"
imf_imports_path = "data/imf_imports.csv"
us_economy_path = "data/us_economy.csv"
cb_policy_rates_path = "data/cb_policy_rates.csv"
global_manufacturing_PMI_path = "data/global_manufacturing_PMI.csv"
im_ex_path = "data/im_ex/"

target_countries = ["VN", "TH", "SG", "ID", "MY", "PH", "BN", "KH", "LA", "MM", "TL", "DE", "FR", "IT", "ES", "NL", "GB", "BE", "AT", "PT", "GR", "FI", "IE", "DK", "SE", "KW", "OM", "BH"]


In [4]:
# 1. REAL GDP GROWTH (Tăng trưởng GDP Thực tế - Hàng Quý 'Q' hoặc Hàng Năm 'A')
print("Fetching Real GDP Growth Data...")
imf_gdp = fetch_imf_data(
    country_dict=country_dict, 
    dataset_code = "IFS",
    metric_suffix="NGDP_R_XDC",
    frequency="A"
).dropna(subset=['value'])

# Tính YoY % trực tiếp trong Python nếu chuỗi dữ liệu gốc là dạng Số tuyệt đối
imf_gdp['value_yoy'] = imf_gdp.groupby('country_name')['value'].pct_change(periods=4) 
imf_gdp.to_csv(imf_gdp_path, index=False)

Fetching Real GDP Growth Data...


Could not load series: {'dataset_code': 'IFS', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'A.VA.NGDP_R_XDC'}
Could not load series: {'dataset_code': 'IFS', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'A.SX.NGDP_R_XDC'}
Could not load series: {'dataset_code': 'IFS', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'A.VE.NGDP_R_XDC'}
Could not load series: {'dataset_code': 'IFS', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'A.UZ.NGDP_R_XDC'}
Could not load series: {'dataset_code': 'IFS', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'A.SUH.NGDP_R_XDC'}


In [4]:
# 2. Unemployment Rate
unemployment_list = []
for country_code in country_dict_3_char.keys():
    df = fetch_series(f"IMF/WEO:latest/{country_code}.LUR.pcent_total_labor_force")
    unemployment_list.append(df)
unemployment_raw = pd.concat(unemployment_list, ignore_index=True)
unemployment_raw.to_csv(imf_unemployment_path)

In [5]:
unemployment_raw

,@frequency,provider_code,dataset_code,dataset_name,series_code,series_name,original_period,period,original_value,value,weo-country,weo-subject,unit,WEO Country,WEO Subject,Unit
0,annual,IMF,WEO:2025-04,World Economic Outlook by countries,VNM.LUR.pcent_total_labor_force,Vietnam – Unemployment rate – Percent of total...,1980,1980-01-01,NA,NaN,VNM,LUR,pcent_total_labor_force,Vietnam,Unemployment rate,Percent of total labor force
1,annual,IMF,WEO:2025-04,World Economic Outlook by countries,VNM.LUR.pcent_total_labor_force,Vietnam – Unemployment rate – Percent of total...,1981,1981-01-01,NA,NaN,VNM,LUR,pcent_total_labor_force,Vietnam,Unemployment rate,Percent of total labor force
2,annual,IMF,WEO:2025-04,World Economic Outlook by countries,VNM.LUR.pcent_total_labor_force,Vietnam – Unemployment rate – Percent of total...,1982,1982-01-01,NA,NaN,VNM,LUR,pcent_total_labor_force,Vietnam,Unemployment rate,Percent of total labor force
3,annual,IMF,WEO:2025-04,World Economic Outlook by countries,VNM.LUR.pcent_total_labor_force,Vietnam – Unemployment rate – Percent of total...,1983,1983-01-01,NA,NaN,VNM,LUR,pcent_total_labor_force,Vietnam,Unemployment rate,Percent of total labor force
4,annual,IMF,WEO:2025-04,World Economic Outlook by countries,VNM.LUR.pcent_total_labor_force,Vietnam – Unemployment rate – Percent of total...,1984,1984-01-01,NA,NaN,VNM,LUR,pcent_total_labor_force,Vietnam,Unemployment rate,Percent of total labor force
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2137,annual,IMF,WEO:2025-04,World Economic Outlook by countries,IRQ.LUR.pcent_total_labor_force,Iraq – Unemployment rate – Percent of total la...,2026,2026-01-01,NA,None,IRQ,LUR,pcent_total_labor_force,Iraq,Unemployment rate,Percent of total labor force
2138,annual,IMF,WEO:2025-04,World Economic Outlook by countries,IRQ.LUR.pcent_total_labor_force,Iraq – Unemployment rate – Percent of total la...,2027,2027-01-01,NA,None,IRQ,LUR,pcent_total_labor_force,Iraq,Unemployment rate,Percent of total labor force
2139,annual,IMF,WEO:2025-04,World Economic Outlook by countries,IRQ.LUR.pcent_total_labor_force,Iraq – Unemployment rate – Percent of total la...,2028,2028-01-01,NA,None,IRQ,LUR,pcent_total_labor_force,Iraq,Unemployment rate,Percent of total labor force
2140,annual,IMF,WEO:2025-04,World Economic Outlook by countries,IRQ.LUR.pcent_total_labor_force,Iraq – Unemployment rate – Percent of total la...,2029,2029-01-01,NA,None,IRQ,LUR,pcent_total_labor_force,Iraq,Unemployment rate,Percent of total labor force


In [6]:
# 3. INDUSTRIAL PRODUCTION INDEX - IPI (Thay đổi dataset_code="IFS")
ipi_raw = fetch_imf_bulk(
    dataset_code="IFS",
    metric_code="AIP_IX", 
    frequency="M", 
    country_list=target_countries
)
if not ipi_raw.empty:
    ipi_raw.dropna(subset=['value']).to_csv(imf_industrial_production_index_path, index=False)

--- Đang tải dữ liệu cho mã: AIP_IX (M) ---
✅ Tải thành công 11153 dòng dữ liệu!


In [7]:
# 4. CPI
imf_cpi = fetch_imf_data(country_dict=country_dict, dataset_code = "CPI", metric_suffix="PCPI_IX", frequency="M").dropna(subset=['value'])
# Show 10 samples
imf_cpi.head(10)
# Export data
imf_cpi.to_csv(imf_cpi_path)

In [8]:
# 5. Central bank policy rates
cb_policy_rates_list = []
for country_code in country_dict.keys():
    cb_policy_rates_df = fetch_series(f"BIS/WS_CBPOL/D.{country_code}")
    cb_policy_rates_list.append(cb_policy_rates_df)

cb_policy_rates = pd.concat(cb_policy_rates_list, ignore_index=True)
cb_policy_rates.to_csv(cb_policy_rates_path)

Could not load series: {'dataset_code': 'WS_CBPOL', 'message': 'Could not load series', 'provider_code': 'BIS', 'series_code': 'D.VN'}
Could not load series: {'dataset_code': 'WS_CBPOL', 'message': 'Could not load series', 'provider_code': 'BIS', 'series_code': 'D.SG'}
Could not load series: {'dataset_code': 'WS_CBPOL', 'message': 'Could not load series', 'provider_code': 'BIS', 'series_code': 'D.BN'}
Could not load series: {'dataset_code': 'WS_CBPOL', 'message': 'Could not load series', 'provider_code': 'BIS', 'series_code': 'D.KH'}
Could not load series: {'dataset_code': 'WS_CBPOL', 'message': 'Could not load series', 'provider_code': 'BIS', 'series_code': 'D.LA'}
Could not load series: {'dataset_code': 'WS_CBPOL', 'message': 'Could not load series', 'provider_code': 'BIS', 'series_code': 'D.MM'}
Could not load series: {'dataset_code': 'WS_CBPOL', 'message': 'Could not load series', 'provider_code': 'BIS', 'series_code': 'D.TL'}
Could not load series: {'dataset_code': 'WS_CBPOL', 'me

In [9]:
# 6. Retail Sales Growth
# DBnomics: OECD (KEI).
retail_sales_raw_set_1 = fetch_series(
    provider_code="OECD", 
    dataset_code="MEI", 
    dimensions={
        "SUBJECT": ["SLRTTO02"],     #  Sales > Retail trade > Total retail trade > Value
        "MEASURE": ["IXEB"],    # US Dollars, monthly level
        "FREQUENCY": ["M"],     # Lấy theo Tháng (Monthly) để monitor cực nhạy
        "LOCATION": ['AUT', 'BEL', 'BGR', 'CHE', 'ESP', 'EST', 'FIN', 'FRA', 'GRC', 'HRV', 'HUN', 'IRL', 'ITA', 'LTU', 'NLD', 'NOR', 'PRT', 'SVK', 'TUR']
    }
)
retail_sales_raw_set_2 = fetch_series(
    provider_code="OECD", 
    dataset_code="MEI", 
    dimensions={
        "SUBJECT": ["SLRTTO02"],
        "MEASURE": ["ML"],
        "FREQUENCY": ["M"],
        "LOCATION": ['CHN', 'RUS', 'USA', 'ZAF']
    }
)
retail_sales_raw = pd.concat([retail_sales_raw_set_1, retail_sales_raw_set_2], ignore_index=True)

In [ ]:
# 7. Global Manufacturing PMI
# DBnomics: WB (World Bank - Pink Sheet Commodity Prices) hoặc IMF.
global_manufacturing_PMI = fetch_series(
    provider_code="OECD", 
    dataset_code="MEI", 
    dimensions={
        "SUBJECT": ["BSCICP02"], 
        "FREQUENCY": ["M"],
    }
)
global_manufacturing_PMI.to_csv(global_manufacturing_PMI_path)

In [9]:
# 8. Merchandise Exports/Imports Growth
# DBnomics: WTO (World Trade Organization) hoặc IMF (Bộ dữ liệu DOT - Direction of Trade Statistics).
im_ex = fetch_series(
    provider_code="IMF", 
    dataset_code="DOT",
    max_nb_series = 53188,
    dimensions={
        "FREQ": ["M"],
        "REF_AREA": ["VN"],
        "INDICATOR": ["TBG_USD"]
    }
)
im_ex

,@frequency,provider_code,dataset_code,dataset_name,series_code,series_name,original_period,period,original_value,value,FREQ,REF_AREA,INDICATOR,COUNTERPART_AREA,Frequency,Reference Area,Indicator,Counterpart Reference Area
0,monthly,IMF,DOT,Direction of Trade Statistics (DOTS),M.VN.TBG_USD.1C_080,"Monthly – Viet Nam – Goods, Value of Trade Bal...",1960-01,1960-01-01,-0.050000,-0.050000,M,VN,TBG_USD,1C_080,Monthly,Viet Nam,"Goods, Value of Trade Balance, US Dollars",Export earnings: fuel
1,monthly,IMF,DOT,Direction of Trade Statistics (DOTS),M.VN.TBG_USD.1C_080,"Monthly – Viet Nam – Goods, Value of Trade Bal...",1960-02,1960-02-01,-0.050000,-0.050000,M,VN,TBG_USD,1C_080,Monthly,Viet Nam,"Goods, Value of Trade Balance, US Dollars",Export earnings: fuel
2,monthly,IMF,DOT,Direction of Trade Statistics (DOTS),M.VN.TBG_USD.1C_080,"Monthly – Viet Nam – Goods, Value of Trade Bal...",1960-04,1960-04-01,-0.100000,-0.100000,M,VN,TBG_USD,1C_080,Monthly,Viet Nam,"Goods, Value of Trade Balance, US Dollars",Export earnings: fuel
3,monthly,IMF,DOT,Direction of Trade Statistics (DOTS),M.VN.TBG_USD.1C_080,"Monthly – Viet Nam – Goods, Value of Trade Bal...",1961-01,1961-01-01,-0.050000,-0.050000,M,VN,TBG_USD,1C_080,Monthly,Viet Nam,"Goods, Value of Trade Balance, US Dollars",Export earnings: fuel
4,monthly,IMF,DOT,Direction of Trade Statistics (DOTS),M.VN.TBG_USD.1C_080,"Monthly – Viet Nam – Goods, Value of Trade Bal...",1961-02,1961-02-01,-0.050000,-0.050000,M,VN,TBG_USD,1C_080,Monthly,Viet Nam,"Goods, Value of Trade Balance, US Dollars",Export earnings: fuel
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,monthly,IMF,DOT,Direction of Trade Statistics (DOTS),M.VN.TBG_USD._X,"Monthly – Viet Nam – Goods, Value of Trade Bal...",2009-08,2009-08-01,127.116666,127.116666,M,VN,TBG_USD,_X,Monthly,Viet Nam,"Goods, Value of Trade Balance, US Dollars",Not allocated/unspecified
236,monthly,IMF,DOT,Direction of Trade Statistics (DOTS),M.VN.TBG_USD._X,"Monthly – Viet Nam – Goods, Value of Trade Bal...",2009-09,2009-09-01,127.116666,127.116666,M,VN,TBG_USD,_X,Monthly,Viet Nam,"Goods, Value of Trade Balance, US Dollars",Not allocated/unspecified
237,monthly,IMF,DOT,Direction of Trade Statistics (DOTS),M.VN.TBG_USD._X,"Monthly – Viet Nam – Goods, Value of Trade Bal...",2009-10,2009-10-01,127.116666,127.116666,M,VN,TBG_USD,_X,Monthly,Viet Nam,"Goods, Value of Trade Balance, US Dollars",Not allocated/unspecified
238,monthly,IMF,DOT,Direction of Trade Statistics (DOTS),M.VN.TBG_USD._X,"Monthly – Viet Nam – Goods, Value of Trade Bal...",2009-11,2009-11-01,127.116666,127.116666,M,VN,TBG_USD,_X,Monthly,Viet Nam,"Goods, Value of Trade Balance, US Dollars",Not allocated/unspecified


In [ ]:
# ==========================================
# 1. CẤU HÌNH & THIẾT LẬP
# ==========================================
# Đặt True khi muốn CẬP NHẬT DỮ LIỆU MỚI
# Đặt False khi bị CRASH/NGẮT MẠNG và muốn CHẠY NỐI TIẾP
FORCE_UPDATE = False  

im_ex_path = "data/im_ex"
os.makedirs(im_ex_path, exist_ok=True)

country_codes = list(country_dict.keys())
CHUNK_SIZE = 10

# ==========================================
# 2. TIẾN HÀNH TẢI DỮ LIỆU THEO CHUNK
# ==========================================
print(f"🚀 Bắt đầu quá trình tải (Chế độ FORCE_UPDATE = {FORCE_UPDATE})...\n")

for i in range(0, len(country_codes), CHUNK_SIZE):
    chunk_codes = country_codes[i : i + CHUNK_SIZE]
    batch_num = (i // CHUNK_SIZE) + 1
    file_path = os.path.join(im_ex_path, f"im_ex_batch_{batch_num}.csv")
    
    # Kiểm tra điều kiện bỏ qua nếu file đã tồn tại và không ép buộc update
    if not FORCE_UPDATE and os.path.exists(file_path):
        print(f"⏩ Nhóm {batch_num} ({len(chunk_codes)} nước) đã có file -> Bỏ qua.")
        continue

    print(f"🔄 Đang tải nhóm {batch_num}/{-(len(country_codes) // -CHUNK_SIZE)} ({len(chunk_codes)} nước): {chunk_codes}...")
    
    try:
        df_chunk = fetch_series(
            provider_code="IMF", 
            dataset_code="DOT",
            max_nb_series=53188,
            dimensions={
                "FREQ": ["M"],
                "REF_AREA": chunk_codes,
                "INDICATOR": ["TBG_USD"]
            }
        )
        
        if df_chunk is not None and not df_chunk.empty:
            df_chunk.to_csv(file_path, index=False, encoding="utf-8-sig")
            print(f"   => ✅ Đã lưu: {file_path}")
        else:
            print(f"   => ⚠️ Nhóm {batch_num} không phản hồi dữ liệu.")
            
    except Exception as e:
        print(f"   => ❌ Lỗi ở nhóm {batch_num}: {e}")

print("\n🎉 Hoàn thành xong bước tải dữ liệu!")

🚀 Bắt đầu quá trình tải (Chế độ FORCE_UPDATE = False)...

🔄 Đang tải nhóm 1/8 (10 nước): ['VN', 'TH', 'SG', 'ID', 'MY', 'PH', 'BN', 'KH', 'LA', 'MM']...
   => ✅ Đã lưu: data/im_ex\im_ex_batch_1.csv
🔄 Đang tải nhóm 2/8 (10 nước): ['TL', 'DE', 'FR', 'IT', 'ES', 'NL', 'GB', 'BE', 'AT', 'PT']...
   => ✅ Đã lưu: data/im_ex\im_ex_batch_2.csv
🔄 Đang tải nhóm 3/8 (10 nước): ['GR', 'FI', 'IE', 'DK', 'SE', 'PL', 'CZ', 'RO', 'HU', 'VA']...
   => ✅ Đã lưu: data/im_ex\im_ex_batch_3.csv
🔄 Đang tải nhóm 4/8 (10 nước): ['UA', 'TR', 'XK', 'US', 'CA', 'MX', 'BR', 'AR', 'CO', 'SR']...
   => ✅ Đã lưu: data/im_ex\im_ex_batch_4.csv
🔄 Đang tải nhóm 5/8 (10 nước): ['SV', 'SX', 'TT', 'UY', 'VC', 'VE', 'SA', 'AE', 'QA', 'KW']...
   => ✅ Đã lưu: data/im_ex\im_ex_batch_5.csv
🔄 Đang tải nhóm 6/8 (10 nước): ['OM', 'BH', 'IQ', 'SY', 'TJ', 'TM', 'UZ', 'YE', 'SN', 'SO']...
   => ✅ Đã lưu: data/im_ex\im_ex_batch_6.csv
🔄 Đang tải nhóm 7/8 (10 nước): ['SS', 'SZ', 'TD', 'TG', 'TN', 'TZ', 'UG', 'ZA', 'ZM', 'ZW']...
   => ✅

In [12]:
# 9. Commodity Price Index
# DBnomics: WB (World Bank - Pink Sheet Commodity Prices) hoặc IMF.

In [13]:
# Timeframe covering Trump baseline through the entirety of the Biden administration
start = datetime.datetime(2000, 1, 1)
end = datetime.datetime(2025, 12, 31)

# Define the exact FRED series codes mapped to your portfolio naming convention
try:
    df_list = []
    for name, code in us_economy_metrics.items():
        print(f"-> Fetching {name} ({code})...")
        # Pull data directly from St. Louis Fed servers
        df = web.DataReader(code, 'fred', start, end)
        # 1. Clean the individual metric's index and rename columns
        df = df.reset_index().rename(columns={'DATE': 'Date', code: 'Value'})
        # 2. Handle frequency alignment: Forward-fill quarterly data before stacking
        # This ensures April and May inherit Q1 data before it gets mixed with monthly metrics
        if name in ['Real_GDP', 'Manufacturing_Investment']:
            # Create a continuous monthly date range to map the quarterly data onto
            monthly_range = pd.date_range(start=start, end=end, freq='MS')
            df = df.set_index('Date').reindex(monthly_range).ffill().reset_index().rename(columns={'index': 'Date'})
        # 3. Add the metadata column so we know which metric this row belongs to
        df['Metric_Name'] = name
        # Reorder columns to look clean: Date | Metric_Name | Value
        df = df[['Date', 'Metric_Name', 'Value']]        
        df_list.append(df)
    print("Consolidating and formatting data frequencies...")
    # Concatenate all tables along the Date axis
    bi_dataset = pd.concat(df_list, axis=0, ignore_index=True)
    # Forward-fill (ffill) the quarterly data (GDP & Investment) so monthly rows aren't blank
    # This prevents relationship errors inside Power BI's model
    bi_dataset = bi_dataset.ffill()
    # Reset index to turn the Date from an index into a normal clean column
    bi_dataset = bi_dataset.reset_index().rename(columns={'DATE': 'Date'})
    # Export to local project directory
    bi_dataset.to_csv(us_economy_path, index=False)
    print(f"Success! Master file saved as '{us_economy_path}'")
    print(bi_dataset.tail(10))

except Exception as e:
    print(f"Pipeline Execution Failed: {e}")

-> Fetching CPI_All_Items (CPIAUCSL)...
-> Fetching Unemployment_Rate (UNRATE)...
-> Fetching Total_Nonfarm_Payrolls (PAYEMS)...
-> Fetching Real_GDP (GDPC1)...
-> Fetching Manufacturing_Investment (C307RX1Q020SBEA)...
Consolidating and formatting data frequencies...
Success! Master file saved as 'data/us_economy.csv'
      index       Date               Metric_Name    Value
1550   1550 2025-03-01  Manufacturing_Investment  145.228
1551   1551 2025-04-01  Manufacturing_Investment  142.245
1552   1552 2025-05-01  Manufacturing_Investment  142.245
1553   1553 2025-06-01  Manufacturing_Investment  142.245
1554   1554 2025-07-01  Manufacturing_Investment  136.654
1555   1555 2025-08-01  Manufacturing_Investment  136.654
1556   1556 2025-09-01  Manufacturing_Investment  136.654
1557   1557 2025-10-01  Manufacturing_Investment  126.551
1558   1558 2025-11-01  Manufacturing_Investment  126.551
1559   1559 2025-12-01  Manufacturing_Investment  126.551
